# MSAI-630 Residency Day 1 — Conditional WGAN-GP (Blond_Hair)

**Course:** Generative AI with Large Language Models (MSAI-630-M41)  
**University of the Cumberlands — PhD in IT (AI Specialization)**  
**Assignment:** Project 1 — Building a Conditional GAN (CGAN)

## Assignment requirements addressed
1. Condition the CGAN on the CelebA **Blond_Hair** attribute (2 classes).
2. Modify `train_step` so generator and critic receive labels in their required formats.
3. Submit **code + generated output** and **interpret** observations / accuracy.

**Graded interpretation:** `../observations.md` (fidelity, diversity, conditioning, and further improvements from Foster Ch. 4).

Adapted from the book notebook:  
https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition/blob/main/notebooks/04_gan/03_cgan/cgan.ipynb


## Why `train_step` must change

| Network | Label format | Shape |
|---------|--------------|-------|
| Generator | Vector one-hot concatenated with noise | `(batch, 2)` |
| Critic | Same label repeated as a spatial map concatenated with the image | `(batch, 64, 64, 2)` |

Passing the generator’s vector label straight into the critic is a shape/format mismatch.  
The fix expands `one_hot_labels` → `image_one_hot_labels` before every critic and gradient-penalty call.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, metrics, optimizers

from notebooks.utils import display, sample_batch

print("TensorFlow", tf.__version__)
print("Conditioning label: Blond_Hair")

## 0. Parameters

In [ ]:
IMAGE_SIZE = 64
CHANNELS = 3
CLASSES = 2
BATCH_SIZE = 64
Z_DIM = 32
LEARNING_RATE = 0.00005
ADAM_BETA_1 = 0.5
ADAM_BETA_2 = 0.9
EPOCHS = 40
STEPS_PER_EPOCH = 5
CRITIC_STEPS = 3
GP_WEIGHT = 10.0
LOAD_MODEL = False
MAX_SAMPLES = 8000
LABEL = "Blond_Hair"

OUTPUT_DIR = ROOT / "output"
CHECKPOINT_DIR = ROOT / "checkpoint"
for d in (OUTPUT_DIR, CHECKPOINT_DIR, ROOT / "models", ROOT / "logs"):
    d.mkdir(parents=True, exist_ok=True)

## 1. Prepare the data

CelebA faces with the **Blond_Hair** attribute. Labels are one-hot encoded to two classes:
- `[1, 0]` → not blond (class 0)
- `[0, 1]` → blond (class 1)

This notebook uses TensorFlow Datasets (`celeb_a`) so paths work outside the book’s Docker `/app/data/...` layout.

In [ ]:
import tensorflow_datasets as tfds

def preprocess_example(example):
    image = example["image"]
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    image = (tf.cast(image, tf.float32) - 127.5) / 127.5
    blond = tf.cast(example["attributes"][LABEL], tf.int32)
    one_hot = tf.one_hot(blond, depth=CLASSES)
    return image, one_hot

ds = tfds.load("celeb_a", split="train", shuffle_files=True)
ds = ds.map(preprocess_example, num_parallel_calls=tf.data.AUTOTUNE)
ds = ds.take(MAX_SAMPLES)
train = (
    ds.shuffle(2048)
      .batch(BATCH_SIZE, drop_remainder=True)
      .prefetch(tf.data.AUTOTUNE)
)

train_sample = sample_batch(train)
display(train_sample, cmap=None)

## 2. Build the GAN

### Critic
Input: RGB image **and** a spatial label map `(64, 64, 2)`.

In [ ]:
critic_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
label_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CLASSES))
x = layers.Concatenate(axis=-1)([critic_input, label_input])
x = layers.Conv2D(64, kernel_size=4, strides=2, padding="same")(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Conv2D(128, kernel_size=4, strides=2, padding="same")(x)
x = layers.LeakyReLU()(x)
x = layers.Dropout(0.3)(x)
x = layers.Conv2D(128, kernel_size=4, strides=2, padding="same")(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Dropout(0.3)(x)
x = layers.Conv2D(128, kernel_size=4, strides=2, padding="same")(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Dropout(0.3)(x)
x = layers.Conv2D(1, kernel_size=4, strides=1, padding="valid")(x)
critic_output = layers.Flatten()(x)

critic = models.Model([critic_input, label_input], critic_output)
critic.summary()

### Generator
Input: noise vector of width `Z_DIM=32` **concatenated** with a 2-class one-hot label.

In [ ]:
generator_input = layers.Input(shape=(Z_DIM,))
label_input = layers.Input(shape=(CLASSES,))
x = layers.Concatenate(axis=-1)([generator_input, label_input])
x = layers.Reshape((1, 1, Z_DIM + CLASSES))(x)
x = layers.Conv2DTranspose(128, kernel_size=4, strides=1, padding="valid", use_bias=False)(x)
x = layers.BatchNormalization(momentum=0.9)(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
x = layers.BatchNormalization(momentum=0.9)(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
x = layers.BatchNormalization(momentum=0.9)(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
x = layers.BatchNormalization(momentum=0.9)(x)
x = layers.LeakyReLU(0.2)(x)
generator_output = layers.Conv2DTranspose(
    CHANNELS, kernel_size=4, strides=2, padding="same", activation="tanh"
)(x)
generator = models.Model([generator_input, label_input], generator_output)
generator.summary()

### ConditionalWGAN — `train_step` label expansion (required change)

```python
# one_hot_labels:              (batch, 2)           -> generator
# image_one_hot_labels:        (batch, 64, 64, 2)   -> critic + GP
image_one_hot_labels = one_hot_labels[:, None, None, :]
image_one_hot_labels = tf.repeat(image_one_hot_labels, repeats=IMAGE_SIZE, axis=1)
image_one_hot_labels = tf.repeat(image_one_hot_labels, repeats=IMAGE_SIZE, axis=2)
```

In [ ]:
class ConditionalWGAN(models.Model):
    def __init__(self, critic, generator, latent_dim, critic_steps, gp_weight):
        super(ConditionalWGAN, self).__init__()
        self.critic = critic
        self.generator = generator
        self.latent_dim = latent_dim
        self.critic_steps = critic_steps
        self.gp_weight = gp_weight

    def compile(self, c_optimizer, g_optimizer):
        super(ConditionalWGAN, self).compile(run_eagerly=True)
        self.c_optimizer = c_optimizer
        self.g_optimizer = g_optimizer
        self.c_wass_loss_metric = metrics.Mean(name="c_wass_loss")
        self.c_gp_metric = metrics.Mean(name="c_gp")
        self.c_loss_metric = metrics.Mean(name="c_loss")
        self.g_loss_metric = metrics.Mean(name="g_loss")

    @property
    def metrics(self):
        return [
            self.c_loss_metric,
            self.c_wass_loss_metric,
            self.c_gp_metric,
            self.g_loss_metric,
        ]

    def gradient_penalty(self, batch_size, real_images, fake_images, image_one_hot_labels):
        alpha = tf.random.normal([batch_size, 1, 1, 1], 0.0, 1.0)
        diff = fake_images - real_images
        interpolated = real_images + alpha * diff

        with tf.GradientTape() as gp_tape:
            gp_tape.watch(interpolated)
            pred = self.critic([interpolated, image_one_hot_labels], training=True)

        grads = gp_tape.gradient(pred, [interpolated])[0]
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]))
        gp = tf.reduce_mean((norm - 1.0) ** 2)
        return gp

    def train_step(self, data):
        real_images, one_hot_labels = data

        # REQUIRED FIX: expand vector labels to spatial maps for the critic
        image_one_hot_labels = one_hot_labels[:, None, None, :]
        image_one_hot_labels = tf.repeat(image_one_hot_labels, repeats=IMAGE_SIZE, axis=1)
        image_one_hot_labels = tf.repeat(image_one_hot_labels, repeats=IMAGE_SIZE, axis=2)

        batch_size = tf.shape(real_images)[0]

        for i in range(self.critic_steps):
            random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))

            with tf.GradientTape() as tape:
                fake_images = self.generator(
                    [random_latent_vectors, one_hot_labels], training=True
                )
                fake_predictions = self.critic(
                    [fake_images, image_one_hot_labels], training=True
                )
                real_predictions = self.critic(
                    [real_images, image_one_hot_labels], training=True
                )
                c_wass_loss = tf.reduce_mean(fake_predictions) - tf.reduce_mean(
                    real_predictions
                )
                c_gp = self.gradient_penalty(
                    batch_size, real_images, fake_images, image_one_hot_labels
                )
                c_loss = c_wass_loss + c_gp * self.gp_weight

            c_gradient = tape.gradient(c_loss, self.critic.trainable_variables)
            self.c_optimizer.apply_gradients(
                zip(c_gradient, self.critic.trainable_variables)
            )

        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))

        with tf.GradientTape() as tape:
            fake_images = self.generator(
                [random_latent_vectors, one_hot_labels], training=True
            )
            fake_predictions = self.critic(
                [fake_images, image_one_hot_labels], training=True
            )
            g_loss = -tf.reduce_mean(fake_predictions)

        gen_gradient = tape.gradient(g_loss, self.generator.trainable_variables)
        self.g_optimizer.apply_gradients(
            zip(gen_gradient, self.generator.trainable_variables)
        )

        self.c_loss_metric.update_state(c_loss)
        self.c_wass_loss_metric.update_state(c_wass_loss)
        self.c_gp_metric.update_state(c_gp)
        self.g_loss_metric.update_state(g_loss)
        return {m.name: m.result() for m in self.metrics}


cgan = ConditionalWGAN(
    critic=critic,
    generator=generator,
    latent_dim=Z_DIM,
    critic_steps=CRITIC_STEPS,
    gp_weight=GP_WEIGHT,
)

if LOAD_MODEL:
    cgan.load_weights(str(CHECKPOINT_DIR / "checkpoint.weights.h5"))


## 3. Train the GAN

In [ ]:
cgan.compile(
    c_optimizer=optimizers.Adam(
        learning_rate=LEARNING_RATE, beta_1=ADAM_BETA_1, beta_2=ADAM_BETA_2
    ),
    g_optimizer=optimizers.Adam(
        learning_rate=LEARNING_RATE, beta_1=ADAM_BETA_1, beta_2=ADAM_BETA_2
    ),
)

model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath=str(CHECKPOINT_DIR / "checkpoint.weights.h5"),
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

class ImageGenerator(callbacks.Callback):
    def __init__(self, num_img, latent_dim):
        self.num_img = num_img
        self.latent_dim = latent_dim
        self.fixed_noise = tf.random.normal(shape=(num_img, latent_dim))

    def on_epoch_end(self, epoch, logs=None):
        zero_label = np.repeat([[1, 0]], self.num_img, axis=0)
        one_label = np.repeat([[0, 1]], self.num_img, axis=0)

        generated_images = self.model.generator([self.fixed_noise, zero_label])
        generated_images = generated_images * 127.5 + 127.5
        display(
            generated_images.numpy(),
            save_to=str(OUTPUT_DIR / f"generated_img_{epoch:03d}_label_0.png"),
            cmap=None,
        )

        generated_images = self.model.generator([self.fixed_noise, one_label])
        generated_images = generated_images * 127.5 + 127.5
        display(
            generated_images.numpy(),
            save_to=str(OUTPUT_DIR / f"generated_img_{epoch:03d}_label_1.png"),
            cmap=None,
        )


In [ ]:
# Short residency-style run (CPU-friendly). Extend EPOCHS for stronger faces.
# Prefer: python train_cgan.py  (produces the recorded submission outputs)
history = cgan.fit(
    train,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=[model_checkpoint_callback, ImageGenerator(num_img=10, latent_dim=Z_DIM)],
)

## 4. Generate images (both classes)

Same noise `z` for both labels — the cleanest visual test that **Blond_Hair** conditioning works.

In [ ]:
z_sample = np.random.normal(size=(10, Z_DIM))

class_label_0 = np.repeat([[1, 0]], 10, axis=0)
imgs0 = cgan.generator.predict([z_sample, class_label_0])
print("Label 0 (not Blond_Hair)")
display(imgs0, save_to=str(OUTPUT_DIR / "final_label_0_not_blond.png"), cmap=None)

class_label_1 = np.repeat([[0, 1]], 10, axis=0)
imgs1 = cgan.generator.predict([z_sample, class_label_1])
print("Label 1 (Blond_Hair)")
display(imgs1, save_to=str(OUTPUT_DIR / "final_label_1_blond.png"), cmap=None)

## 5. Observations about output and accuracy

**Graded write-up (paste / upload for LMS):** `observations.md`

That single document covers the course prompt:

1. **Fidelity** — do samples resemble faces?
2. **Diversity** — avoid mode collapse?
3. **Conditioning** — does Blond_Hair control work? (shared-`z` comparison)
4. **Further improvements** — Foster Ch.4 approaches applied after the short baseline (balanced classes, cache, graph mode, 2,000 WGAN-GP steps)

### Recorded primary run
- Conditional WGAN-GP + required spatial label expansion in `train_step`
- Class-balanced Blond_Hair, graph-mode, 80×25 = **2,000** steps (~24 min CPU)
- Figures: `output/final_label_0_not_blond.png`, `final_label_1_blond.png`, `final_shared_z_comparison.png`
- Baseline short run: `output/baseline_run/`

### Quick verdict
- Fidelity: improved but still soft/noisy
- Diversity: no total mode collapse
- Conditioning: **yes** — shared-`z` darker vs lighter hair when the label flips
